In [0]:
/FileStore/tables/sales_csv.txt
/FileStore/tables/menu_csv.txt

In [0]:
from pyspark.sql.types import *
schema=StructType([
    StructField("product_id",IntegerType(),True),
    StructField('customer_id',StringType(),True),
    StructField('order_date',DateType(),True),
    StructField('location',StringType(),True),
    StructField('source_order',StringType(),True)
])

sales_df=spark.read.csv("/FileStore/tables/sales_csv.txt",schema=schema)

# sales_df=spark.read.format("csv").option("inferschema","true").schema(schema).load("/FileStore/tables/sales_csv.txt")

# sales_df.show()

display(sales_df)

In [0]:
from pyspark.sql.functions import month,year,quarter
sales_df=sales_df.withColumn("order_year",year(sales_df.order_date))
sales_df=sales_df.withColumn("order_month",month(sales_df.order_date))
sales_df=sales_df.withColumn("quarter",quarter(sales_df.order_date))
display(sales_df)

In [0]:
schema2=StructType([
    StructField('product_id',IntegerType(),True),
    StructField('product_name',StringType(),True),
    StructField('price',StringType(),True)
])

menu_df=spark.read.format('csv').option("inferschema","true").schema(schema2).load("/FileStore/tables/menu_csv.txt")

menu_df.show()

In [0]:
total_amount_spent=sales_df.join(menu_df,sales_df.product_id==menu_df.product_id,"inner")\
    .groupBy("customer_id").agg({'price':'sum'})

display(total_amount_spent.orderBy('customer_id'))

In [0]:
food_spend=(sales_df.join(menu_df,sales_df.product_id==menu_df.product_id,"inner")\
    .groupBy('product_name').agg({'price':'sum'})).orderBy('product_name')
display(food_spend)

In [0]:
month_spend=(sales_df.join(menu_df,'product_id')\
    .groupBy('order_month').agg({'price':'sum'})).orderBy('order_month')
display(month_spend)

In [0]:
year_spend=(sales_df.join(menu_df,'product_id')\
    .groupBy('order_year').agg({'price':'sum'})).orderBy('order_year')
display(year_spend)

In [0]:
quarter_spend=(sales_df.join(menu_df,'product_id')\
    .groupBy('quarter').agg({'price':'sum'})).orderBy('quarter')
display(quarter_spend)

In [0]:
prod_count=sales_df.join(menu_df,'product_id')\
    .groupBy('product_name').count()
display(prod_count)

In [0]:
from pyspark.sql.functions import *
prod_count = (
    sales_df.join(menu_df, 'product_id')
    .groupBy('product_name')
    .agg(count('product_name').alias('count'))
    .orderBy("count", ascending=False)
    .limit(1)
)

display(prod_count)

In [0]:
freq=(sales_df.filter(sales_df.source_order=='Restaurant')).groupBy('customer_id').agg(countDistinct('order_date'))
display(freq)

In [0]:
country_spend = (
    sales_df.join(menu_df, 'product_id')
    .groupBy('location')
    .agg(sum('price').alias('sales'))
    .orderBy('sales')
)
display(country_spend)

In [0]:
source_spend = (
    sales_df.join(menu_df, 'product_id')
    .groupBy('source_order')
    .agg(sum('price').alias('sales'))
    .orderBy('sales')
)
display(source_spend)